# 📈 Stock Market Prediction with Deep Learning
### LSTM · GRU · BiLSTM + Attention · Monte Carlo Uncertainty · Technical Indicators

This notebook builds a complete stock price forecasting pipeline:
- **Live data** fetched from Yahoo Finance for any ticker
- **20 technical indicator features**: RSI, MACD, Bollinger Bands, ATR, OBV, moving averages, and more
- **Three architectures** trained and compared: Vanilla LSTM · GRU · Bidirectional LSTM + Attention
- **Uncertainty estimation** via Monte Carlo Dropout (90% confidence intervals)
- **Five evaluation metrics**: RMSE · MAE · MAPE · R² · Directional Accuracy
- **Interactive Plotly charts** for predictions, residuals, and model comparison

In [ ]:
# Install dependencies (uncomment if running in Colab)
# !pip install yfinance plotly ta tensorflow scikit-learn -q

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from data.data_pipeline import fetch_and_prepare
from models.models import (
    build_vanilla_lstm, build_gru, build_bilstm_attention,
    get_callbacks, monte_carlo_predict
)
from evaluation.evaluation import (
    compute_metrics, print_metrics_table,
    plot_training_history, plot_predictions,
    plot_residuals, plot_metrics_radar, plot_feature_correlation
)
import numpy as np
import pandas as pd

## 1 · Configuration

In [ ]:
TICKER      = 'MSFT'      # ← change to any Yahoo Finance ticker
START_DATE  = '2015-01-01'
SEQ_LEN     = 60          # look-back window in trading days
EPOCHS      = 100
BATCH_SIZE  = 32
MC_ITER     = 50          # Monte Carlo dropout iterations

## 2 · Fetch Data & Engineer Features

In [ ]:
data = fetch_and_prepare(ticker=TICKER, start=START_DATE, sequence_length=SEQ_LEN)

X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']
scaler_y        = data['scaler_y']
dates_test      = data['dates_test']

y_test_price = scaler_y.inverse_transform(y_test.reshape(-1,1)).squeeze()
input_shape  = (X_train.shape[1], X_train.shape[2])

print(f'Input shape  : {input_shape}')
print(f'Train samples: {len(X_train)}')
print(f'Test  samples: {len(X_test)}')

## 3 · Feature Correlation

In [ ]:
fig = plot_feature_correlation(data['raw_df'], data['feature_names'])
fig.show()

## 4 · Train All Three Models

In [ ]:
model_builders = {
    'VanillaLSTM':      lambda: build_vanilla_lstm(input_shape),
    'GRU':              lambda: build_gru(input_shape),
    'BiLSTM_Attention': lambda: build_bilstm_attention(input_shape),
}

trained_models, histories, predictions, mc_bands, metrics_list = {}, {}, {}, {}, []

for name, builder in model_builders.items():
    print(f'\n=== Training {name} ===')
    model = builder()
    hist = model.fit(
        X_train, y_train,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        validation_split=0.1,
        callbacks=get_callbacks(),
        verbose=1, shuffle=False,
    )
    trained_models[name] = model
    histories[name]      = hist.history

    y_pred_scaled = model.predict(X_test, verbose=0).squeeze()
    y_pred_price  = scaler_y.inverse_transform(y_pred_scaled.reshape(-1,1)).squeeze()
    predictions[name] = y_pred_price

    mc_m, mc_lo, mc_hi = monte_carlo_predict(model, X_test, MC_ITER)
    mc_bands[name] = (
        scaler_y.inverse_transform(mc_m.reshape(-1,1)).squeeze(),
        scaler_y.inverse_transform(mc_lo.reshape(-1,1)).squeeze(),
        scaler_y.inverse_transform(mc_hi.reshape(-1,1)).squeeze(),
    )

    m = compute_metrics(y_test_price, y_pred_price, label=name)
    metrics_list.append(m)
    print(f'  RMSE={m["RMSE"]:.3f}  MAE={m["MAE"]:.3f}  MAPE={m["MAPE (%)"]:.2f}%  R²={m["R²"]:.4f}  DirAcc={m["Dir. Acc. (%)"]:.1f}%')

## 5 · Results Table

In [ ]:
print_metrics_table(metrics_list)
pd.DataFrame(metrics_list)

## 6 · Training Loss Curves

In [ ]:
plot_training_history(histories).show()

## 7 · Predictions with Uncertainty Bands

In [ ]:
plot_predictions(dates_test, y_test_price, predictions, mc_bands, TICKER).show()

## 8 · Residuals

In [ ]:
plot_residuals(dates_test, y_test_price, predictions).show()

## 9 · Model Radar Comparison

In [ ]:
plot_metrics_radar(metrics_list).show()